# Benchmark stage (полный конвейер)

Один проход по всем хешам, **со своим бюджетом на каждый хеш**. Атаки берутся из конфига (нет конфига → не гоним; NeuralHash → только оптимизированная атака). Сохраняются все артефакты (метрики, история сходимости на пару, исходные/целевые/атакованные изображения) — повторно гонять не нужно.

Этапы ноутбука:
1. прогон **original**-конфига и **optimized**-конфига (+ evolved) в два `RunStore`;
2. сводки результатов;
3. визуализация: сходимость, source/target/attacked, интерактивный выбор пар, экспорт PNG для диплома;
4. генерация LaTeX-таблиц диплома из артефактов.

Повтор с тем же `run_name` **дозапускает** недостающее (resume).

In [ ]:
%matplotlib inline
import os
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import sys

os.environ['NEURALHASH_PROVIDER'] = 'cpu'
os.environ.setdefault('NEURALHASH_INTRA_THREADS', '1')
os.environ.setdefault('NEURALHASH_INTER_THREADS', '1')

os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')

DATASET = "evohash/data/dataset_v1_full.zip"  # relative to the repo parent (cwd is set below)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "evohash" / "__init__.py").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

from evohash.stage4_benchmark.runner import BenchmarkConfig, run_benchmark
from evohash.stage4_benchmark.artifacts import RunStore
from evohash.stage4_benchmark import viz
from evohash.stage4_benchmark import tables as T

## Общая конфигурация
Свой бюджет на каждый хеш. PhotoDNA вне Windows пропускается автоматически. LPIPS считается (нужен для таблиц); поставь `compute_lpips=False` для скорости.

In [ ]:
ORIGINAL  = "evohash/results/attack_configs.json"
OPTIMIZED = "evohash/results/attack_configs_optimized.json"

COMMON = dict(
    dataset_zip=DATASET,         # из setup-ячейки (сейчас dataset_v1_full.zip)
    split="public",              # "public" (30 пар) | "secret" (100 пар, финальная валидация)
    output_root="evohash/outputs/stage_bench",   # явный путь; артефакты -> output_root/<run_name>
    hashes=("phash", "pdq", "photodna", "neuralhash"),
    max_pairs=None,                 # None = все пары сплита; меньше -> быстрый тест
    budgets={"phash": 10_000, "pdq": 10_000, "photodna": 20_000, "neuralhash": 30_000,},
    default_budget=10_000,
    compute_lpips=True,
    backend="process",           # process на всех хешах | thread | sequential
    workers=10,                   # ручной потолок воркеров (None = авто по cpu_count); dict -> на хеш
)

## 1a. Прогон original-конфига (baseline)
Без evolved (эволюционные кандидаты гоняем только в optimized-проходе).

In [ ]:
orig = run_benchmark(BenchmarkConfig(
    optimized_config_path=ORIGINAL, include_evolved=False,
    run_name=f"original_{COMMON['split']}", **COMMON,
), progress=True)

## 1b. Прогон optimized-конфига + evolved

In [ ]:
opt = run_benchmark(BenchmarkConfig(
    optimized_config_path=OPTIMIZED, include_evolved=True,
    run_name=f"optimized_{COMMON['split']}", **COMMON,
), progress=True)

## 2. Сводки
Позже без прогона (имена со сплитом): `orig = RunStore.open(f"outputs/stage_bench/original_{COMMON['split']}")`, `opt = RunStore.open(f"outputs/stage_bench/optimized_{COMMON['split']}")`.

In [ ]:
from IPython.display import display

frames = T.diploma_display_frames(orig, opt)
for caption, frame in frames.items():
    display(T.style_diploma_frame(frame, caption=caption))


In [ ]:
# Optional LaTeX export of the same diploma-style tables.
tex = T.latex_diploma_tables(orig, opt)
paths = T.write_tables(f"outputs/tables_{COMMON['split']}", **tex)
print("written:", *[p.name for p in paths.values()])


## 3. Визуализация

In [ ]:
HASH, ATTACK = "phash", "pHash_OpenEvolve"
# viz.convergence(opt, HASH, ATTACK); plt.show()

In [ ]:
# viz.show_triple(opt, HASH, ATTACK, opt.pairs(HASH, ATTACK)[0], amplify=10); plt.show()

In [ ]:
# viz.gallery(opt, HASH, ATTACK, opt.pairs(HASH, ATTACK)[:4], amplify=12); plt.show()

## Интерактивный выбор пар + экспорт PNG для диплома
Дропдауны hash/attack/pair, кнопка `export PNG`. (Нужен `ipywidgets`.)

In [ ]:
viz.browse(opt)

In [ ]:
# программный экспорт выбранных троек
for pid in opt.pairs(HASH, ATTACK)[:3]:
    viz.export_triple(opt, HASH, ATTACK, pid, f"outputs/diploma_figs/{HASH}_{ATTACK}_{pid}.png")
print("saved -> outputs/diploma_figs/")

## 4. LaTeX-таблицы диплома
Генераторы возвращают готовое `table`-окружение с тем же `\caption`/`\label`, что в `diplom.tex`. Пишутся в `outputs/tables/*.tex` — можно `\input{...}` или скопировать. `diplom.tex` ничего не подтягивает сам, пока ты не вставишь.

In [ ]:
# tex = {f"baseline_optimized_{h}": T.latex_baseline_optimized(orig, opt, h) for h in opt.hashes()}
# tex["evolved_results"] = T.latex_evolved_results(opt)
# tex["best_optimized_vs_evolved"] = T.latex_best_optimized_vs_evolved(opt)

# paths = T.write_tables("outputs/tables", **tex)
# print("written:", *[p.name for p in paths.values()])
# print()
# print(tex["best_optimized_vs_evolved"])   # превью

Таблицы по порогам / защите (`tab:phash_threshold_sweep`, consistency / purification) строятся в `Defense_phash.ipynb` из его DataFrame через `T.latex_threshold_sweep(...)` и `T.dataframe_to_latex(...)`.